In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [15]:
from waymo_agent.notebook_imports import *

In [3]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from kret_sandbox.VIS import dtt

In [4]:
config = EnvConfig()
plt_cfg = PlotConfig()

In [5]:
env = RideShareEnv(config, plt_cfg)
G = env.G
cfg = env.config

Assigned lambda values to nodes. Total lambda: 0.1475 (target: 0.1564)


/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/simulation/generate_obs_state.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  z = (vehicles.f_available.sum() / requests.f_awaiting_price.sum()) - 1


In [6]:
s, info = env.reset()

/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/simulation/generate_obs_state.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  z = (vehicles.f_available.sum() / requests.f_awaiting_price.sum()) - 1


In [7]:
def get_obvs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides

In [8]:
veh, req, rides = get_obvs_tuple(env)
veh.shape, req.shape, rides.shape

((2, 6), (25, 21), (2, 18))

In [9]:
action_rnd = env.action_space.sample()
dispatches = pd.DataFrame({"dispatch": action_rnd["dispatch"]})
prices = pd.DataFrame({"prices": action_rnd["prices"]})
reposition = pd.DataFrame(action_rnd["reposition"], columns=["x", "y"])
action_rnd["dispatch"] = np.full(action_rnd["dispatch"].shape, cfg.no_action_id, dtype=np.int32)
action_rnd["reposition"] = np.array([[-0.1, -0.1], [0.1, 0.1]], dtype=np.float32)
env.action_space.contains(action_rnd)

True

## Pre-Step

In [10]:
# env.config.max_new_requests_per_step = 0

In [11]:
action_rnd["dispatch"][2] = 0
action_rnd["dispatch"][3] = 1
# action["dispatch"] = np.full(action["dispatch"].shape, cfg.no_action_id, dtype=np.int32)
env._validate_action(action_rnd)
action_rnd["dispatch"]

array([-1, -1,  0,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1], dtype=int32)

In [12]:
dtt([req[req.f_valid], veh, rides], n=12, how="head")

request_id 
 request_dt 
 cust_id 
 cust_bias 
 cust_temperature 
 est_cost 
 price 
 max_wait_time 
 wait_time 
 status 
 pickup_node_id 
 pickup_x_norm 
 pickup_y_norm 
 dropoff_node_id 
 dropoff_x_norm 
 dropoff_y_norm 
 route_nodes 
 curr_start_node 
 curr_end_node 
 route_dist_on_edge 
 distance_meters 
 
 
 
 
 
 
 
 
 vehicle_id 
 loc_x_norm 
 loc_y_norm 
 battery 
 status 
 ride_id 
 
 
 
 
 0 
 0 
 -0.025 
 0.407 
 0.849 
 0 
 -1 
 
 
 1 
 1 
 0.301 
 0.323 
 0.775 
 0 
 -1 
 
 
 
 
 
 
 ride_id 
 vehicle_id 
 pickup_node 
 pickup_x_norm 
 pickup_y_norm 
 dropoff_node 
 dropoff_x_norm 
 dropoff_y_norm 
 price 
 est_cost 
 total_trip_distance_meters 
 trip_distance_remaining_meters 
 pickup_distance_remaining_meters 
 route_nodes 
 curr_start_node 
 curr_end_node 
 route_dist_on_edge 
 is_complete 
 
 
 
 
 0 
 -1 
 0 
 -1 
 -1.0 
 -1.0 
 -1 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 [] 
 -1 
 -1 
 -1.0 
 False 
 
 
 1 
 -1 
 1 
 -1 
 -1.0 
 -1.0 
 -1 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 -1.0 
 [] 
 -1 
 -1 
 -1.0 
 False

In [13]:
dtt([env.breadcrumbs, env.DiscardedRequests])

ValueError: No objects to concatenate

# Take Step

In [14]:
env.num_vehicles

2